# Zomato year-wise feedback analysis

Year-wise positive/negative comparison for Google News, Trustpilot, Reddit and Google Play.

## Load the database

In [3]:
from pathlib import Path
import sqlite3
import pandas as pd
import plotly.express as px
from IPython.display import display

def parse_date(values):
    raw = values.astype('string').str.strip()
    parsed = pd.to_datetime(raw, errors='coerce', utc=True, format='mixed')
    numeric = pd.to_numeric(raw, errors='coerce')
    fallback = pd.to_datetime(numeric, errors='coerce', unit='s', utc=True)
    return parsed.fillna(fallback)

PLOTLY_TEMPLATE = 'plotly_white'
def style_chart(fig, title):
    fig.update_layout(title={'text': title, 'x': 0.02, 'xanchor': 'left'}, template=PLOTLY_TEMPLATE, height=430, margin={'l': 55, 'r': 25, 't': 70, 'b': 50}, font={'family': 'Arial', 'size': 13, 'color': '#1f2937'}, legend={'orientation': 'h', 'y': 1.08, 'x': 0, 'title': ''}, plot_bgcolor='#ffffff', paper_bgcolor='#ffffff', bargap=0.22)
    fig.update_xaxes(showgrid=False, title=None, tickmode='linear')
    fig.update_yaxes(title=None, gridcolor='#e5e7eb', separatethousands=True)
    fig.update_traces(textposition='outside', hovertemplate='%{x}<br>%{fullData.name}: %{y:,}<extra></extra>')
    return fig
ROOT = Path.cwd()
DB = ROOT / 'data' / 'db' / 'Zomato.db'
OUT = ROOT / 'output'
OUT.mkdir(exist_ok=True)

with sqlite3.connect(DB) as con:
    items = pd.read_sql('SELECT * FROM content_items', con)
    links = pd.read_sql('SELECT content_hash, source, raw_id FROM source_tracking', con)
    news_queries = pd.read_sql('SELECT article_id, search_query, article_date FROM google_news_query_tracking', con)

items['event_date'] = parse_date(items['published_date'].fillna(items['created_at']))
items['year'] = items['event_date'].dt.year.astype('Int64')
links = links.drop_duplicates(['content_hash', 'source'])
df = links.merge(items, on='content_hash', how='left')
print(f'items={len(items):,} | sources={links.source.nunique()}')

items=905,641 | sources=4


In [4]:
display(items[['content_type', 'title', 'rating', 'published_date']].head())

,content_type,title,rating,published_date
0,article,When discounts cost dearly: Why Bengaluru rest...,NaN,"Sun, 16 Aug 2026 03:30:00 GMT"
1,article,"After Zepto, Food Safety Lens On Zomato, Inspe...",NaN,"Wed, 12 Aug 2026 05:16:18 GMT"
2,article,"NPS without a fixed minimum: How Zomato, Swigg...",NaN,"Mon, 17 Aug 2026 01:30:04 GMT"
3,article,"Zomato now wants discounts to convert demand, ...",NaN,"Fri, 14 Aug 2026 15:27:01 GMT"
4,article,"I quit Swiggy, Zomato for a month and cooked m...",NaN,"Fri, 14 Aug 2026 01:30:00 GMT"


In [5]:
display(items.dtypes.rename('dtype').to_frame().head())

,dtype
content_hash,object
content_type,object
title,object
content,object
rating,float64


## Data-quality checks

In [6]:
checks = pd.Series({
    'four active sources': links['source'].nunique() == 4,
    'unique content keys': items['content_hash'].is_unique,
    'source links have master rows': df['content_type'].notna().all(),
    'ratings are in range': items['rating'].dropna().between(1, 5).all(),
    'usable event dates': items['event_date'].notna().mean() >= 0.90,
})
hard_checks = checks.drop('usable event dates')
if not hard_checks.all():
    raise ValueError('Structural data-quality check failed.')
print(f'quality_passed={hard_checks.all()} | invalid_dates={int(items["event_date"].isna().sum()):,}')

quality_passed=True | invalid_dates=0


In [7]:
display(links.groupby('source')['content_hash'].nunique().rename('unique_items').to_frame().head())

,unique_items
source,
google_news,568
google_play,900159
reddit,4761
trustpilot,153


In [8]:
display(items['event_date'].isna().rename('invalid_event_date').to_frame().head())

,invalid_event_date
0,False
1,False
2,False
3,False
4,False


## Trustpilot and Google Play: separate star-rating charts

In [9]:
rating_rows = df[df['source'].isin(['trustpilot', 'google_play'])].dropna(subset=['rating', 'year']).copy()
rating_rows['positive'] = rating_rows['rating'].ge(4)
rating_rows['negative'] = rating_rows['rating'].lt(4)
rating_summary = rating_rows.groupby(['year', 'source']).agg(
    positive_count=('positive', 'sum'), negative_count=('negative', 'sum'), total_feedback=('rating', 'size')
).reset_index()
rating_summary['positive_ratio'] = (rating_summary['positive_count'] / rating_summary['total_feedback']).round(3)
rating_summary['negative_ratio'] = (rating_summary['negative_count'] / rating_summary['total_feedback']).round(3)
rating_summary['positive_negative_ratio'] = (rating_summary['positive_count'] / rating_summary['negative_count'].replace(0, float('nan'))).round(2)
display(rating_summary.head())

overall_ratings = rating_rows.groupby('year').agg(
    positive_count=('positive', 'sum'), negative_count=('negative', 'sum'), total_feedback=('rating', 'size')
).reset_index()
overall_ratings['positive_ratio'] = (overall_ratings['positive_count'] / overall_ratings['total_feedback']).round(3)
overall_ratings['negative_ratio'] = (overall_ratings['negative_count'] / overall_ratings['total_feedback']).round(3)

,year,source,positive_count,negative_count,total_feedback,positive_ratio,negative_ratio,positive_negative_ratio
0,2014,trustpilot,0,2,2,0.000,1.000,0.00
1,2015,trustpilot,2,4,6,0.333,0.667,0.50
2,2016,trustpilot,0,2,2,0.000,1.000,0.00
3,2018,trustpilot,4,7,11,0.364,0.636,0.57
4,2019,trustpilot,2,4,6,0.333,0.667,0.50


In [10]:
display(overall_ratings.head())

,year,positive_count,negative_count,total_feedback,positive_ratio,negative_ratio
0,2014,0,2,2,0.000,1.000
1,2015,2,4,6,0.333,0.667
2,2016,0,2,2,0.000,1.000
3,2018,4,7,11,0.364,0.636
4,2019,2,4,6,0.333,0.667


In [11]:
plot_data = rating_summary[rating_summary["source"].eq("google_play")].melt("year", ["positive_count", "negative_count"], var_name="feedback", value_name="count")
plot_data["feedback"] = plot_data["feedback"].map({"positive_count": "Positive (4-5 stars)", "negative_count": "Negative (1-3 stars)"})
fig = px.bar(plot_data, x="year", y="count", color="feedback", barmode="group", text_auto=True, title="Google Play feedback by year", template=PLOTLY_TEMPLATE, color_discrete_map={"Positive (4-5 stars)": "#16a34a", "Negative (1-3 stars)": "#dc2626"})
fig = style_chart(fig, "Google Play feedback by year")
fig.show()


In [12]:
plot_data = rating_summary[rating_summary["source"].eq("trustpilot")].melt("year", ["positive_count", "negative_count"], var_name="feedback", value_name="count")
plot_data["feedback"] = plot_data["feedback"].map({"positive_count": "Positive (4-5 stars)", "negative_count": "Negative (1-3 stars)"})
fig = px.bar(plot_data, x="year", y="count", color="feedback", barmode="group", text_auto=True, title="Trustpilot feedback by year", template=PLOTLY_TEMPLATE, color_discrete_map={"Positive (4-5 stars)": "#16a34a", "Negative (1-3 stars)": "#dc2626"})
fig = style_chart(fig, "Trustpilot feedback by year")
fig.show()


## Google News: broad coverage versus complaints

In [13]:
news_queries['event_date'] = parse_date(news_queries['article_date'])
news_queries['year'] = news_queries['event_date'].dt.year.astype('Int64')
news_queries['query'] = news_queries['search_query'].str.strip().str.lower()
news_queries = news_queries.drop_duplicates(['article_id', 'query'])
broad = news_queries[news_queries['query'].eq('zomato')].groupby('year')['article_id'].nunique().rename('broad_count')
complaints = news_queries[news_queries['query'].eq('zomato complaint')].groupby('year')['article_id'].nunique().rename('complaint_count')
news_summary = pd.concat([broad, complaints], axis=1).fillna(0).reset_index()
news_summary['complaint_ratio'] = (news_summary['complaint_count'] / news_summary['broad_count'].replace(0, float('nan'))).round(3)
display(news_summary.head())

news_plot = news_summary.melt('year', ['broad_count', 'complaint_count'], var_name='query', value_name='articles')
news_plot['query'] = news_plot['query'].map({'broad_count': 'All Zomato articles', 'complaint_count': 'Zomato complaint articles'})
fig = px.bar(news_plot, x='year', y='articles', color='query', barmode='group', text_auto=True, title='Google News coverage and complaints', template=PLOTLY_TEMPLATE, color_discrete_map={'All Zomato articles': '#2563eb', 'Zomato complaint articles': '#dc2626'})
fig = style_chart(fig, 'Google News coverage and complaints')
fig.show()

,year,broad_count,complaint_count,complaint_ratio
0,2026,136.0,27,0.199
1,2021,0.0,1,NaN
2,2022,0.0,3,NaN
3,2024,0.0,17,NaN
4,2025,0.0,38,NaN


## Reddit: complaint posts and comments

In [14]:
reddit = df[(df['source'] == 'reddit') & df['year'].notna()].copy()
reddit['text'] = reddit['title'].fillna('') + ' ' + reddit['content'].fillna('')
complaint_words = r'complaint|refund|late|delay|bad|issue|problem|support|cancel|wrong order|missing order|poor service'
reddit['is_complaint'] = reddit['text'].str.lower().str.contains(complaint_words, regex=True, na=False)
reddit['kind'] = reddit['content_type'].map({'post': 'posts', 'comment': 'comments'}).fillna('other')
reddit_summary = reddit.groupby('year').agg(
    feedback_count=('content_hash', 'nunique'), complaint_count=('is_complaint', 'sum')
).reset_index()
reddit_summary['non_complaint_feedback'] = reddit_summary['feedback_count'] - reddit_summary['complaint_count']
reddit_summary['complaint_rate'] = (reddit_summary['complaint_count'] / reddit_summary['feedback_count'].replace(0, float('nan'))).round(3)
reddit_kind = reddit[reddit['is_complaint']].groupby(['year', 'kind'])['content_hash'].nunique().unstack(fill_value=0).reset_index()
reddit_summary = reddit_summary.merge(reddit_kind, on='year', how='left')
display(reddit_summary.head())

reddit_plot = reddit_summary.melt('year', ['non_complaint_feedback', 'complaint_count'], var_name='feedback', value_name='count')
reddit_plot['feedback'] = reddit_plot['feedback'].map({'non_complaint_feedback': 'Other feedback', 'complaint_count': 'Complaint feedback'})
fig = px.bar(reddit_plot, x='year', y='count', color='feedback', barmode='group', text_auto=True, title='Reddit feedback', template=PLOTLY_TEMPLATE, color_discrete_map={'Other feedback': '#16a34a', 'Complaint feedback': '#dc2626'})
fig = style_chart(fig, 'Reddit feedback')
fig.show()

,year,feedback_count,complaint_count,non_complaint_feedback,complaint_rate,comments,posts
0,2012,2,0,2,0.000,NaN,NaN
1,2014,6,0,6,0.000,NaN,NaN
2,2015,22,0,22,0.000,NaN,NaN
3,2016,5,0,5,0.000,NaN,NaN
4,2017,19,3,16,0.158,2.0,1.0


## Merged year-wise comparison

In [15]:
rated_compare = overall_ratings.assign(source='Trustpilot + Google Play', metric='4-5 stars vs 1-3 stars')
r_compare = rated_compare[['year', 'source', 'positive_count', 'negative_count', 'total_feedback', 'positive_ratio', 'negative_ratio', 'metric']]
n_compare = news_summary.assign(source='Google News', positive_count=news_summary['broad_count'] - news_summary['complaint_count'], negative_count=news_summary['complaint_count'], total_feedback=news_summary['broad_count'], positive_ratio=1 - news_summary['complaint_ratio'], negative_ratio=news_summary['complaint_ratio'], metric='non-complaint coverage proxy vs complaint coverage')
n_compare = n_compare[['year', 'source', 'positive_count', 'negative_count', 'total_feedback', 'positive_ratio', 'negative_ratio', 'metric']]
rd_compare = reddit_summary.assign(source='Reddit', positive_count=reddit_summary['non_complaint_feedback'], negative_count=reddit_summary['complaint_count'], total_feedback=reddit_summary['feedback_count'], positive_ratio=1 - reddit_summary['complaint_rate'], negative_ratio=reddit_summary['complaint_rate'], metric='non-complaint feedback proxy vs complaint feedback')
rd_compare = rd_compare[['year', 'source', 'positive_count', 'negative_count', 'total_feedback', 'positive_ratio', 'negative_ratio', 'metric']]
yearly_comparison = pd.concat([r_compare, n_compare, rd_compare], ignore_index=True).sort_values(['year', 'source'])
display(yearly_comparison.head())

overall_volume = yearly_comparison.groupby('year')[['positive_count', 'negative_count', 'total_feedback']].sum().reset_index()
overall_volume['positive_ratio'] = (overall_volume['positive_count'] / overall_volume['total_feedback']).round(3)
overall_volume['negative_ratio'] = (overall_volume['negative_count'] / overall_volume['total_feedback']).round(3)
yearly_comparison.to_csv(OUT / 'yearly_feedback_comparison.csv', index=False)
rating_summary.to_csv(OUT / 'yearly_rating_source_summary.csv', index=False)
news_summary.to_csv(OUT / 'yearly_google_news_summary.csv', index=False)
reddit_summary.to_csv(OUT / 'yearly_reddit_summary.csv', index=False)
overall_volume.to_csv(OUT / 'yearly_all_source_volume.csv', index=False)

,year,source,positive_count,negative_count,total_feedback,positive_ratio,negative_ratio,metric
17,2012,Reddit,2.0,0,2.0,1.000,0.000,non-complaint feedback proxy vs complaint feed...
18,2014,Reddit,6.0,0,6.0,1.000,0.000,non-complaint feedback proxy vs complaint feed...
0,2014,Trustpilot + Google Play,0.0,2,2.0,0.000,1.000,4-5 stars vs 1-3 stars
19,2015,Reddit,22.0,0,22.0,1.000,0.000,non-complaint feedback proxy vs complaint feed...
1,2015,Trustpilot + Google Play,2.0,4,6.0,0.333,0.667,4-5 stars vs 1-3 stars


In [16]:
display(overall_volume.head())

,year,positive_count,negative_count,total_feedback,positive_ratio,negative_ratio
0,2012,2.0,0,2.0,1.000,0.000
1,2014,6.0,2,8.0,0.750,0.250
2,2015,24.0,4,28.0,0.857,0.143
3,2016,5.0,2,7.0,0.714,0.286
4,2017,16.0,3,19.0,0.842,0.158


## Business answer

In [20]:
summary = []
if not overall_ratings.empty:
    row = overall_ratings.loc[overall_ratings['negative_ratio'].idxmax()]
    summary.append(f" | Ratings negative peak={int(row['year'])} ({row['negative_ratio']:.1%})\n")
if not news_summary.empty:
    row = news_summary.loc[news_summary['complaint_ratio'].idxmax()]
    summary.append(f"News complaint peak={int(row['year'])} ({row['complaint_ratio']:.1%})\n")
if not reddit_summary.empty:
    row = reddit_summary.loc[reddit_summary['complaint_rate'].idxmax()]
    summary.append(f"Reddit complaint peak={int(row['year'])} ({row['complaint_rate']:.1%})\n")
print(' | '.join(summary))

 | Ratings negative peak=2014 (100.0%)
 | News complaint peak=2026 (19.9%)
 | Reddit complaint peak=2018 (21.3%)

